### Pre-processing par seuil bof: For HSV, hue range is [0,179], saturation range is [0,255], and value range is [0,255]

In [ ]:
from scipy.signal import find_peaks

def annotate_peaks(ax, hist, color, height=0.01):
    peaks, _ = find_peaks(hist, height=height)
    for p in peaks:
        ax.annotate(f"{int(p)}", xy=(p, hist[p]), xytext=(p, hist[p] + max(hist)*0.02),
                    textcoords='data', ha='center', fontsize=8, color=color, rotation=45)

# Dossier des images
img_dir = "dataset_project_iapr2025/train/"
img_files = sorted([f for f in os.listdir(img_dir) if f.lower().endswith(".jpg")])

# Itérer sur toutes les images
for filename in img_files:
    print(f"Traitement de : {filename}")
    path = os.path.join(img_dir, filename)

    # === Ton pipeline (inchangé) ===
    img = cv.imread(path)
    img_rgb = cv.cvtColor(img, cv.COLOR_BGR2RGB)

    # Masquage noir
    img_noir = img.copy()
    img_noir_rgb = cv.cvtColor(img_noir, cv.COLOR_BGR2RGB)

    # mask BACKGROUND
    mask_rgb = (
        (img_noir_rgb[:, :, 0] >= 105) & (img_noir_rgb[:, :, 0] <= 195) &
        (img_noir_rgb[:, :, 1] >= 105) & (img_noir_rgb[:, :, 1] <= 195) &
        (img_noir_rgb[:, :, 2] >= 105) & (img_noir_rgb[:, :, 2] <= 195)
    )

    img_noir_hsv = cv.cvtColor(img_noir_rgb, cv.COLOR_RGB2HSV)

    mask_hsv = (
        (((img_noir_hsv[:, :, 0] >= 110) & (img_noir_hsv[:, :, 0] <= 120)) |
        ((img_noir_hsv[:, :, 1] >= 0) & (img_noir_hsv[:, :, 1] <= 50))) |
        ((img_noir_hsv[:, :, 2] >= 120) & (img_noir_hsv[:, :, 2] <= 180))
    )

    mask_total = mask_rgb & mask_hsv
    img_noir_rgb[mask_total] = [0, 0, 0]
    mask_choco = np.where(mask_total, 0, 255).astype(np.uint8)

    # Histogrammes
    img_hsv = cv.cvtColor(img_rgb, cv.COLOR_RGB2HSV)
    r_hist = cv.calcHist([img_rgb], [0], mask_choco, [256], [0, 256]).flatten()
    g_hist = cv.calcHist([img_rgb], [1], mask_choco, [256], [0, 256]).flatten()
    b_hist = cv.calcHist([img_rgb], [2], mask_choco, [256], [0, 256]).flatten()
    h_hist = cv.calcHist([img_hsv], [0], mask_choco, [180], [0, 180]).flatten()
    s_hist = cv.calcHist([img_hsv], [1], mask_choco, [256], [0, 256]).flatten()
    v_hist = cv.calcHist([img_hsv], [2], mask_choco, [256], [0, 256]).flatten()

    # Figure 1
    fig, axs = plt.subplots(2, 2, figsize=(14, 14))
    axs[0, 0].imshow(img_rgb)
    axs[0, 0].set_title(f"Image originale - {filename}")
    axs[0, 0].axis("off")

    axs[0, 1].imshow(img_noir_rgb)
    axs[0, 1].set_title("Image après correction noire")
    axs[0, 1].axis("off")

    axs[1, 0].plot(r_hist, color='r', label='R')
    axs[1, 0].plot(g_hist, color='g', label='G')
    axs[1, 0].plot(b_hist, color='b', label='B')
    axs[1, 0].set_title("Histogrammes RGB du mask")
    axs[1, 0].legend()
    annotate_peaks(axs[1, 0], r_hist, color='r')
    annotate_peaks(axs[1, 0], g_hist, color='g')
    annotate_peaks(axs[1, 0], b_hist, color='b')

    axs[1, 1].plot(h_hist, color='purple', label='H')
    axs[1, 1].plot(s_hist, color='orange', label='S')
    axs[1, 1].plot(v_hist, color='gray', label='V')
    axs[1, 1].set_title("Histogrammes HSV du mask")
    axs[1, 1].legend()
    annotate_peaks(axs[1, 1], h_hist, color='purple')
    annotate_peaks(axs[1, 1], s_hist, color='orange')
    annotate_peaks(axs[1, 1], v_hist, color='gray')

    plt.tight_layout()
    plt.show()
